In [5]:
"""
ibge_coleta.py
==============
Coleta os endpoints IBGE de prioridade Alta definidos em:
  fontes_covariaveis_municipais.md

Fontes cobertas:
  1. IBGE Localidades   — /api/v1/localidades/municipios
  2. IBGE Censo 2022    — SIDRA tabelas 9606, 9605, 9514  (via apisidra)
"""

from locale import D_FMT
import time
import requests
import pandas as pd

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def get_json(url: str, params: dict = None, label: str = "") -> dict | list | None:
    """GET com tratamento de erro e log simples."""
    try:
        resp = requests.get(url, params=params, timeout=90)
        resp.raise_for_status()
        print(f"  ✔ {label or url}")
        return resp.json()
    except requests.HTTPError as e:
        print(f"  ✘ {label} — HTTP {e.response.status_code}")
    except Exception as e:
        print(f"  ✘ {label} — {e}")
    return None


# ══════════════════════════════════════════════
# 1. IBGE LOCALIDADES
# ══════════════════════════════════════════════

def coletar_localidades() -> pd.DataFrame:
    """
    Retorna todos os ~5.570 municípios com:
    id, nome, uf_sigla, uf_nome, regiao_imediata,
    regiao_intermediaria, macroregiao_sigla
    """
    print("\n[1/3] Coletando Localidades…")
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    data = get_json(url, label="municipios")
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame([
        {
            "id_municipio":         m["id"],
            "nome_municipio":       m["nome"],
            "uf_sigla":             m["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"],
            "uf_nome":              m["regiao-imediata"]["regiao-intermediaria"]["UF"]["nome"],
            "regiao_imediata_id":   m["regiao-imediata"]["id"],
            "regiao_imediata_nome": m["regiao-imediata"]["nome"],
            "regiao_interm_id":     m["regiao-imediata"]["regiao-intermediaria"]["id"],
            "regiao_interm_nome":   m["regiao-imediata"]["regiao-intermediaria"]["nome"],
            "macroregiao_sigla":    m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["sigla"],
            "macroregiao_nome":     m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["nome"],
        }
        for m in data
    ])
    print(f"     → {len(df)} municípios carregados")
    return df


# ══════════════════════════════════════════════
# 2. IBGE CENSO 2022 — SIDRA
# ══════════════════════════════════════════════

# Tabelas definidas no documento:
#   9606 — Domicílios com internet por município
#   9605 — Rendimento médio domiciliar per capita
#   9514 — População por sexo e faixa etária
#
# Endpoint padrão apisidra:
#   /values/t/{tabela}/n6/all/v/all/p/last

SIDRA_BASE = "https://apisidra.ibge.gov.br/values"

TABELAS_CENSO = {
    "9606": "Domicilios_com_internet",
    "9605": "Rendimento_medio_domiciliar_percapita",
    "9514": "Populacao_sexo_faixa_etaria",
}


def _parse_sidra(data: list) -> pd.DataFrame:
    """
    A API SIDRA retorna uma lista onde o índice 0 é o cabeçalho
    e os demais são registros.
    """
    if not data or len(data) < 2:
        return pd.DataFrame()
    header = list(data[0].keys())
    columns_names = list(data[0].values())
    rows   = data[1:]
    df = pd.DataFrame(rows, columns=header)
    df.columns = columns_names

    return df


def coletar_censo_sidra() -> dict[str, pd.DataFrame]:
    """
    Retorna um dict  {nome_tabela: DataFrame}
    para cada tabela do Censo 2022 listada acima.
    """
    print("\n[2/3] Coletando Censo 2022 via SIDRA…")
    resultados = {}

    for tabela, nome in TABELAS_CENSO.items():
        url = f"{SIDRA_BASE}/t/{tabela}/n6/all/v/all/p/last"
        data = get_json(url, label=f"SIDRA tabela {tabela} — {nome}")

        if data:
            df = _parse_sidra(data)
            print(f"     → {len(df)} linhas")
            resultados[nome] = df
        else:
            resultados[nome] = pd.DataFrame()

        time.sleep(0.5)   # respeita rate-limit da API

    return resultados


In [19]:
localidades = coletar_localidades()
localidades.head(1)



[1/3] Coletando Localidades…
  ✔ municipios
     → 5571 municípios carregados


,id_municipio,nome_municipio,uf_sigla,uf_nome,regiao_imediata_id,regiao_imediata_nome,regiao_interm_id,regiao_interm_nome,macroregiao_sigla,macroregiao_nome
0,1100015,Alta Floresta D'Oeste,RO,Rondônia,110005,Cacoal,1102,Ji-Paraná,N,Norte


In [3]:
import json

# ── JSON bruto — Localidades ──────────────────────────────────────────────────
_raw_loc = get_json(
    "https://servicodados.ibge.gov.br/api/v1/localidades/municipios",
    label="municipios (raw)",
)

print(f"Tipo da resposta : {type(_raw_loc)}")
print(f"Total de itens   : {len(_raw_loc)}")
print(f"Tipo de cada item: {type(_raw_loc[0])}")
print(f"Chaves de nível 1: {list(_raw_loc[0].keys())}")
print("\n── Primeiro item — JSON bruto completo ──")
print(json.dumps(_raw_loc[0], ensure_ascii=False, indent=2))

  ✔ municipios (raw)
Tipo da resposta : <class 'list'>
Total de itens   : 5571
Tipo de cada item: <class 'dict'>
Chaves de nível 1: ['id', 'nome', 'microrregiao', 'regiao-imediata']

── Primeiro item — JSON bruto completo ──
{
  "id": 1100015,
  "nome": "Alta Floresta D'Oeste",
  "microrregiao": {
    "id": 11006,
    "nome": "Cacoal",
    "mesorregiao": {
      "id": 1102,
      "nome": "Leste Rondoniense",
      "UF": {
        "id": 11,
        "sigla": "RO",
        "nome": "Rondônia",
        "regiao": {
          "id": 1,
          "sigla": "N",
          "nome": "Norte"
        }
      }
    }
  },
  "regiao-imediata": {
    "id": 110005,
    "nome": "Cacoal",
    "regiao-intermediaria": {
      "id": 1102,
      "nome": "Ji-Paraná",
      "UF": {
        "id": 11,
        "sigla": "RO",
        "nome": "Rondônia",
        "regiao": {
          "id": 1,
          "sigla": "N",
          "nome": "Norte"
        }
      }
    }
  }
}


In [6]:
import json, time

# ── JSON bruto — SIDRA (data[0] = cabeçalho, data[1] = primeiro registro) ────

for tabela, nome in TABELAS_CENSO.items():
    url = f"{SIDRA_BASE}/t/{tabela}/n6/all/v/all/p/last"
    _raw = get_json(url, label=f"SIDRA {tabela} (raw)")

    print(f"\n{'═'*60}")
    print(f"  Tabela {tabela} — {nome}")
    print(f"{'═'*60}")
    print(f"Total de itens (cabeçalho + registros): {len(_raw)}")
    print(f"Tipo de cada item                     : {type(_raw[0])}")

    print("\n── data[0] — mapa de cabeçalho (chave interna → nome legível) ──")
    print(json.dumps(_raw[0], ensure_ascii=False, indent=2))

    print("\n── data[1] — primeiro registro real (chaves internas) ──")
    print(json.dumps(_raw[1], ensure_ascii=False, indent=2))

    time.sleep(0.5)

  ✔ SIDRA 9606 (raw)

════════════════════════════════════════════════════════════
  Tabela 9606 — Domicilios_com_internet
════════════════════════════════════════════════════════════
Total de itens (cabeçalho + registros): 11141
Tipo de cada item                     : <class 'dict'>

── data[0] — mapa de cabeçalho (chave interna → nome legível) ──
{
  "NC": "Nível Territorial (Código)",
  "NN": "Nível Territorial",
  "MC": "Unidade de Medida (Código)",
  "MN": "Unidade de Medida",
  "V": "Valor",
  "D1C": "Município (Código)",
  "D1N": "Município",
  "D2C": "Variável (Código)",
  "D2N": "Variável",
  "D3C": "Ano (Código)",
  "D3N": "Ano",
  "D4C": "Sexo (Código)",
  "D4N": "Sexo",
  "D5C": "Cor ou raça (Código)",
  "D5N": "Cor ou raça",
  "D6C": "Idade (Código)",
  "D6N": "Idade"
}

── data[1] — primeiro registro real (chaves internas) ──
{
  "NC": "6",
  "NN": "Município",
  "MC": "45",
  "MN": "Pessoas",
  "V": "21494",
  "D1C": "1100015",
  "D1N": "Alta Floresta D'Oeste - RO",
  "D

In [16]:
resultados = coletar_censo_sidra()


[2/3] Coletando Censo 2022 via SIDRA…
  ✔ SIDRA tabela 9606 — Domicilios_com_internet
     → 11140 linhas
  ✔ SIDRA tabela 9605 — Rendimento_medio_domiciliar_percapita
     → 11140 linhas
  ✔ SIDRA tabela 9514 — Populacao_sexo_faixa_etaria
     → 11140 linhas


In [20]:
resultados['Domicilios_com_internet'].head(1)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Sexo (Código),Sexo,Cor ou raça (Código),Cor ou raça,Idade (Código),Idade
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,6794,Total,95251,Total,100362,Total


In [21]:
resultados['Rendimento_medio_domiciliar_percapita'].head(1)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Cor ou raça (Código),Cor ou raça
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,95251,Total


In [22]:
resultados['Populacao_sexo_faixa_etaria'].head(1)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Sexo (Código),Sexo,Forma de declaração da idade (Código),Forma de declaração da idade,Idade (Código),Idade
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,6794,Total,113635,Total,100362,Total


In [ ]:
# ══════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════

if __name__ == "__main__":

    # 1. Localidades ──────────────────────────
    df_localidades = coletar_localidades()
    print(df_localidades.head(3).to_string())

    # 2. Censo 2022 / SIDRA ───────────────────
    censo = coletar_censo_sidra()
    for nome, df in censo.items():
        print(f"\n── {nome} ({len(df)} linhas) ──")
        print(df.head(3).to_string())